# T12 — Demo reproducible del artefacto

Construcción inicial S8 para la tesis **Detección de anomalías conductuales en transacciones de billeteras digitales mediante modelos de aprendizaje automático no supervisado**.

## Diseño actualizado tras observaciones del examen parcial
- baseline construido en warm-up legítimo y congelado;
- 12 features (6 raw/contextuales + 6 individualizadas);
- train limpio sin anomalías;
- IF y AE como modelos centrales; **LOF como baseline formal**;
- búsqueda de hiperparámetros usando validation;
- early stopping del AE con holdout temporal normal dentro de train;
- threshold F1 como referencia + presupuestos top-k;
- Recall frente a intensidad continua D y bootstrap pareado.


In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))
from src.simulator import SimulationConfig
from run_demo import run

# Ejecución reproducible del notebook.
cfg = SimulationConfig(n_users=120, seed=42)
run(cfg, Path("outputs_notebook"), tuning="quick", bootstrap_iterations=200)



=== T12 EXPERIMENTO COMPLETADO ===
{
  "seed": 42,
  "n_users": 120,
  "n_transactions": 12000,
  "warmup_transactions": 3600,
  "train_transactions": 5880,
  "train_fit_transactions": 4704,
  "train_early_stop_transactions": 1176,
  "validation_transactions": 1200,
  "test_transactions": 1320,
  "train_anomalies": 0,
  "validation_anomalies": 24,
  "test_anomalies": 26,
  "anomaly_rate_validation": 0.02,
  "anomaly_rate_test": 0.019696969696969695,
  "raw_features": 6,
  "hybrid_features": 12,
  "baseline_policy": "Frozen after legitimate warm-up; no post-warm-up observation updates it.",
  "training_policy": "Clean normal-only train by design; anomalies only in validation/test.",
  "early_stopping_policy": "Normal-only temporal holdout carved from train; external validation is not used for AE stopping.",
  "threshold_policy": "F1-optimal threshold chosen on external validation as reference; top-k alert budgets are also reported.",
  "model_policy": "Isolation Forest and Autoencoder 

In [2]:
import pandas as pd
metrics = pd.read_csv("outputs_notebook/metrics_global.csv")
metrics[["model","feature_set","precision","recall","f1","pr_auc","fp","fn"]]

,model,feature_set,precision,recall,f1,pr_auc,fp,fn
0,Isolation Forest,raw,0.032258,0.153846,0.053333,0.034578,120,22
1,Autoencoder,raw,1.000000,0.038462,0.074074,0.082360,0,25
2,LOF baseline,raw,0.058824,0.038462,0.046512,0.040101,16,25
3,Isolation Forest,raw+individualizadas,0.444444,0.153846,0.228571,0.189903,5,22
4,Autoencoder,raw+individualizadas,0.310345,0.346154,0.327273,0.394409,20,17
5,LOF baseline,raw+individualizadas,0.642857,0.346154,0.450000,0.411907,5,17


In [3]:
recall_d = pd.read_csv("outputs_notebook/recall_vs_D.csv")
recall_d


,model,D_left,D_right,D_mid,mean_D,n_anomalies,recall,mean_score,feature_set
0,if,0.1,0.2,0.15,0.196522,1,0.000000,0.408013,raw+individualizadas
1,if,0.2,0.3,0.25,0.230394,4,0.000000,0.459546,raw+individualizadas
2,if,0.3,0.4,0.35,0.351763,3,0.000000,0.462798,raw+individualizadas
3,if,0.4,0.5,0.45,0.445603,3,0.000000,0.469662,raw+individualizadas
4,if,0.5,0.6,0.55,0.500000,2,0.500000,0.605519,raw+individualizadas
5,if,0.6,0.7,0.65,0.612734,1,0.000000,0.527586,raw+individualizadas
6,if,0.7,0.8,0.75,0.741864,1,0.000000,0.536489,raw+individualizadas
7,if,0.8,0.9,0.85,0.859634,3,0.333333,0.563591,raw+individualizadas
8,if,0.9,1.0,0.95,0.993459,8,0.250000,0.560086,raw+individualizadas
9,ae,0.1,0.2,0.15,0.196522,1,0.000000,0.264679,raw+individualizadas


In [4]:
pd.read_csv("outputs_notebook/precision_at_k.csv")


,alert_budget,top_k,score_cutoff,precision_at_k,recall_at_k,true_positives_at_k,model,feature_set
0,0.005,7,0.641478,0.000000,0.000000,0,Isolation Forest,raw
1,0.010,14,0.631986,0.000000,0.000000,0,Isolation Forest,raw
2,0.020,27,0.612733,0.037037,0.038462,1,Isolation Forest,raw
3,0.050,66,0.585995,0.045455,0.115385,3,Isolation Forest,raw
4,0.005,7,1.982526,0.142857,0.038462,1,Autoencoder,raw
5,0.010,14,1.545751,0.214286,0.115385,3,Autoencoder,raw
6,0.020,27,1.285552,0.111111,0.115385,3,Autoencoder,raw
7,0.050,66,0.985509,0.045455,0.115385,3,Autoencoder,raw
8,0.005,7,1.974061,0.000000,0.000000,0,LOF baseline,raw
9,0.010,14,1.596104,0.071429,0.038462,1,LOF baseline,raw


In [5]:
pd.read_csv("outputs_notebook/paired_bootstrap_pr_auc.csv")


,delta_pr_auc,ci95_low,ci95_high,n_bootstrap_valid,comparison,A,B
0,0.204506,0.020801,0.359094,200,AE_hybrid_vs_IF_hybrid,Autoencoder | raw+individualizadas,Isolation Forest | raw+individualizadas
1,-0.017498,-0.057351,0.026139,200,AE_hybrid_vs_LOF_hybrid,Autoencoder | raw+individualizadas,LOF baseline | raw+individualizadas
2,-0.222003,-0.397497,-0.055559,200,IF_hybrid_vs_LOF_hybrid,Isolation Forest | raw+individualizadas,LOF baseline | raw+individualizadas
3,0.155325,0.044318,0.292428,200,IF_hybrid_vs_raw,Isolation Forest | raw+individualizadas,Isolation Forest | raw
4,0.312049,0.113022,0.499479,200,AE_hybrid_vs_raw,Autoencoder | raw+individualizadas,Autoencoder | raw
5,0.371806,0.198037,0.542731,200,LOF_hybrid_vs_raw,LOF baseline | raw+individualizadas,LOF baseline | raw


> Estas métricas son evidencia de construcción/levantamiento y no constituyen por sí solas el resultado final de tesis. La corrida definitiva se ejecuta con cinco semillas, tuning completo y bootstrap de 1 000 remuestreos.